# Predicting survival from titanic crash using naive bayes classifier

Credit: https://www.youtube.com/watch?v=PPeaRc-r1OI&list=PLeo1K3hjS3uvCeTYTeyfe0-rN5r8zn9rw&index=15 <br>
Code: https://github.com/codebasics/py/blob/master/ML/14_naive_bayes/14_naive_bayes_1_titanic_survival_prediction.ipynb <br>

Importing basic libraries 

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('titanic.csv')
df.head()

,PassengerId,Name,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Survived
0,1,"Braund, Mr. Owen Harris",3,male,22.0,1,0,A/5 21171,7.2500,NaN,S,0
1,2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,female,38.0,1,0,PC 17599,71.2833,C85,C,1
2,3,"Heikkinen, Miss. Laina",3,female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1
3,4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,female,35.0,1,0,113803,53.1000,C123,S,1
4,5,"Allen, Mr. William Henry",3,male,35.0,0,0,373450,8.0500,NaN,S,0


Droping all the feature columns which are not required to make the survival decision (*i.e. target*)

In [3]:
inputs = df.drop(['PassengerId','Name','SibSp','Parch','Ticket','Cabin','Embarked','Survived'], axis='columns')
inputs.head()

,Pclass,Sex,Age,Fare
0,3,male,22.0,7.2500
1,1,female,38.0,71.2833
2,3,female,26.0,7.9250
3,1,female,35.0,53.1000
4,3,male,35.0,8.0500


In [4]:
target = df.Survived
# target.head()

**Check if any column of input is empty or null**

In [5]:
# np.where(pd.isnull(inputs))
# np.where(pd.isna(inputs))
inputs.isnull().sum()

Pclass      0
Sex         0
Age       177
Fare        0
dtype: int64

In [6]:
inputs.columns[inputs.isna().any()]

Index(['Age'], dtype='object')

In [7]:
inputs['Age'] = inputs['Age'].fillna(inputs['Age'].mean())

In [8]:
inputs.columns[inputs.isna().any()]

Index([], dtype='object')

Since, the feature column **sex** is text, we need to convert into numeral by using **onehotencoder**

In [9]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [10]:
# inputs_le = inputs
inputs.Sex = le.fit_transform(inputs.Sex)
inputs.head()

,Pclass,Sex,Age,Fare
0,3,1,22.0,7.2500
1,1,0,38.0,71.2833
2,3,0,26.0,7.9250
3,1,0,35.0,53.1000
4,3,1,35.0,8.0500


In [11]:
X = inputs[['Pclass','Sex','Age','Fare']].values
X[0:5]

array([[ 3.    ,  1.    , 22.    ,  7.25  ],
       [ 1.    ,  0.    , 38.    , 71.2833],
       [ 3.    ,  0.    , 26.    ,  7.925 ],
       [ 1.    ,  0.    , 35.    , 53.1   ],
       [ 3.    ,  1.    , 35.    ,  8.05  ]])

In [12]:
y = target.values
y[0:5]

array([0, 1, 1, 1, 0], dtype=int64)

In [13]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [14]:
ct_ohe = ColumnTransformer([('Sex',OneHotEncoder(),[1])],remainder='passthrough')
ct_ohe

ColumnTransformer(remainder='passthrough',
                  transformers=[('Sex', OneHotEncoder(), [1])])

In [15]:
X = ct_ohe.fit_transform(X)
X[0:5]

array([[ 0.    ,  1.    ,  3.    , 22.    ,  7.25  ],
       [ 1.    ,  0.    ,  1.    , 38.    , 71.2833],
       [ 1.    ,  0.    ,  3.    , 26.    ,  7.925 ],
       [ 1.    ,  0.    ,  1.    , 35.    , 53.1   ],
       [ 0.    ,  1.    ,  3.    , 35.    ,  8.05  ]])

In [16]:
X = X[:,1:] # consider entire rows and from column number 1 into the X matrix
X[0:5]

array([[ 1.    ,  3.    , 22.    ,  7.25  ],
       [ 0.    ,  1.    , 38.    , 71.2833],
       [ 0.    ,  3.    , 26.    ,  7.925 ],
       [ 0.    ,  1.    , 35.    , 53.1   ],
       [ 1.    ,  3.    , 35.    ,  8.05  ]])

In [17]:
# inputs = X

In [18]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3)

**Gaussuian Naive bayes (GNB)** *assumes that each parameter (also called features or predictors) has an independent capacity of predicting the output variable. The combination of the prediction for all parameters is the final prediction, that returns a probability of the dependent variable to be classified in each group. The final classification is assigned to the group with the higher probability.*

In [19]:
from sklearn.naive_bayes import GaussianNB
model = GaussianNB()

In [20]:
model.fit(X_test, y_test)

GaussianNB()

In [21]:
model.score(X_test, y_test)

0.753731343283582

In [22]:
X_test[0:10]
# (1:male,0:female, Pclass, Age, Fare)

array([[  1.    ,   1.    ,  35.    ,  26.2875],
       [  1.    ,   1.    ,  36.    ,  26.2875],
       [  1.    ,   3.    ,  14.    ,  46.9   ],
       [  1.    ,   1.    ,  24.    , 247.5208],
       [  0.    ,   3.    ,  41.    ,  39.6875],
       [  1.    ,   3.    ,  44.    ,   8.05  ],
       [  1.    ,   3.    ,  28.    ,  15.85  ],
       [  0.    ,   2.    ,  22.    ,  29.    ],
       [  1.    ,   1.    ,  35.    , 512.3292],
       [  1.    ,   3.    ,  32.    ,   7.8958]])

In [23]:
y_test[0:10]
# 1:survived,0:not survived

array([1, 1, 0, 0, 0, 0, 0, 1, 1, 0], dtype=int64)

In [24]:
model.predict(X_test[0:10])

array([0, 0, 0, 1, 1, 0, 0, 1, 1, 0], dtype=int64)

In [25]:
model.score(X_test, y_test)

0.753731343283582

In [26]:
from sklearn.model_selection import cross_val_score
cv = cross_val_score(GaussianNB(), X_train, y_train, cv=5)
cv

array([0.792     , 0.816     , 0.792     , 0.76612903, 0.75806452])

In [27]:
cv.mean()

0.7848387096774194

In [28]:
cross_val_score(GaussianNB(), X_test, y_test, cv=5)

array([0.66666667, 0.77777778, 0.81481481, 0.81132075, 0.66037736])

In [29]:
cross_val_score(GaussianNB(), X_test, y_test, cv=5).mean()

0.7461914744933613